# OCR + Image Fusion Classifier Evaluation

Evaluates a rule-based OCR + image fusion classifier against the image-only baseline
on the same v1 train-dev held-out set. Produces directly comparable accuracy/error numbers
so the improvement (or lack thereof) is measurable.

**Methods compared:**
1. Image-only baseline (ResNet-50, dropout-regularized v1 checkpoint)
2. + OCR override on strong keyword match (always)
3. + OCR override on strong match, only when image confidence < 0.6
4. + OCR override on strong match, only when image confidence < 0.7

---
## 1. Setup

In [33]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import json, random, time
from pathlib import Path
from PIL import Image

import sys
sys.path.insert(0, str(Path.cwd().parent))

from classifier.dataset import ProductDataset
from app.services.extraction import CATEGORY_KEYWORDS, classify_category

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [34]:
STRONG_TERMS = {
    "skincare": {
        "serum", "essence", "ampoule", "booster", "sunscreen", "sunblock",
        "cleanser", "face wash", "micellar", "cleansing oil", "cleansing balm",
        "toner", "toning", "facial mist", "eye cream", "lip balm", "face mask",
        "sheet mask", "clay mask", "body lotion", "body cream", "body oil",
        "body butter", "hand cream", "shower gel", "body wash", "hand soap",
        "liquid soap", "bar soap", "shaving", "shaving foam", "shaving gel",
        "deodorant", "antiperspirant", "gel douche", "masque visage", "savon",
        "lotion corporelle", "creme mains", "soin corps", "lait corps",
        "nettoyant", "mousse \u00e0 raser", "mousse a raser",
    },
    "haircare": {
        "shampoo", "conditioner", "co-wash", "cleansing conditioner",
        "hair mask", "hair treatment", "hair serum", "hair oil", "hair spray",
        "hairspray", "hair mousse", "styling mousse", "hair gel", "pomade",
        "hair cream", "leave-in", "hair wax", "hair paste", "scalp serum",
        "dandruff", "shampooing", "apr\u00e8s-shampooing", "apres-shampoing",
        "masque cheveux", "huile cheveux", "gel coiffant", "s\u00e9rum cheveux",
        "laque", "soin cheveux", "soins cheveux", "cr\u00e8me colorante",
    },
    "makeup": {
        "mascara", "eyeliner", "eye liner", "eyeshadow", "eye shadow",
        "lipstick", "lip gloss", "lip liner", "concealer", "foundation",
        "blush", "bronzer", "nail polish", "vernis \u00e0 ongles", "vernis a ongles",
        "fond de teint", "bb cream", "cc cream", "primer", "highlighter",
        "setting spray", "setting powder", "pressed powder", "loose powder",
        "eyebrow", "brow gel", "brow pencil", "lip plumper", "lip stain",
        "false lashes", "maquillage", "rouge \u00e0 l\u00e8vres", "rouge a levres",
        "fard \u00e0 paupi\u00e8res", "fard a paupieres", "anticernes", "correcteur",
        "gloss", "micellar water", "makeup remover", "cleansing wipe",
    },
}


def match_detail(raw_text):
    """Return (category, keyword, is_strong) mirroring classify_category's pass order."""
    if not raw_text:
        return None, None, None
    lower = raw_text.lower()

    HAIRCARE_CREME = {
        "creme colorante", "cr\u00e8me colorante",
        "creme decolorante", "cr\u00e8me d\u00e9colorante", "creme d\u00e9colorante",
        "creme de coiffage", "cr\u00e8me de coiffage",
        "creme fixante", "cr\u00e8me fixante",
    }
    for term in HAIRCARE_CREME:
        if term in lower:
            return "haircare", term, True

    MAKEUP_UNAMBIGUOUS = {
        "mascara", "eyeliner", "eye liner", "eyeshadow", "eye shadow",
        "lipstick", "lip gloss", "lip liner", "concealer", "foundation",
        "blush", "bronzer", "nail polish", "vernis \u00e0 ongles", "vernis a ongles",
        "fond de teint",
    }
    for term in MAKEUP_UNAMBIGUOUS:
        if term in lower:
            return "makeup", term, True

    best = (None, None, False)
    first_hit = None
    for category, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw in lower:
                is_strong = kw in STRONG_TERMS.get(category, set())
                if first_hit is None:
                    first_hit = (category, kw, is_strong)
                if is_strong and best[0] is None:
                    best = (category, kw, True)
    if best[0] is not None:
        return best
    return first_hit if first_hit is not None else (None, None, None)

---
## 2. Load v1 Train-Dev Split

Identical to `train_classifier.ipynb`: same `ProductDataset(raw_v1_clean, split="train")`,
same `stratified_split(seed=42)`, same `train_dev_idx`. This is the held-out set where
the image-only baseline has 28.42% error.

In [35]:
DATA_DIR = Path.cwd().parent / "data" / "raw_v1_clean"
CKPT_DIR = Path.cwd().parent / "data" / "models"
MANIFEST_DIR = Path.cwd().parent / "data" / "manifests"

train_ds = ProductDataset(DATA_DIR, split="train")

def stratified_split(dataset, dev_pct=0.15, seed=42):
    """Return (train_indices, dev_indices) with stratified class distribution."""
    class_indices = {}
    for i, (_, label) in enumerate(dataset.samples):
        class_indices.setdefault(label, []).append(i)
    rng = random.Random(seed)
    train_idx, dev_idx = [], []
    for label, indices in class_indices.items():
        rng.shuffle(indices)
        n_dev = max(1, int(len(indices) * dev_pct))
        dev_idx.extend(indices[:n_dev])
        train_idx.extend(indices[n_dev:])
    return train_idx, dev_idx

train_idx, train_dev_idx = stratified_split(train_ds)
train_dev_ds = Subset(train_ds, train_dev_idx)
train_dev_loader = DataLoader(train_dev_ds, batch_size=32, shuffle=False)

CLASS_NAMES = train_ds.classes
name_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx_to_name = {i: name for i, name in enumerate(CLASS_NAMES)}

print(f"Classes: {CLASS_NAMES}")
print(f"Train-dev size: {len(train_dev_ds)}")

# Build filename -> (path, true_label_name) mapping for the train-dev set
train_dev_lookup = {}
for idx in train_dev_idx:
    path, label = train_ds.samples[idx]
    fname = Path(path).name
    train_dev_lookup[fname] = (Path(path), CLASS_NAMES[label])
print(f"Unique images in train-dev: {len(train_dev_lookup)}")

# Verify stratification
td_counts = {cls: 0 for cls in CLASS_NAMES}
for idx in train_dev_idx:
    td_counts[CLASS_NAMES[train_ds.samples[idx][1]]] += 1
print("\nTrain-dev class distribution:")
for cls, cnt in td_counts.items():
    print(f"  {cls}: {cnt} ({cnt/len(train_dev_ds)*100:.1f}%)")

Classes: ['haircare', 'makeup', 'skincare']
Train-dev size: 1073
Unique images in train-dev: 1073

Train-dev class distribution:
  haircare: 261 (24.3%)
  makeup: 85 (7.9%)
  skincare: 727 (67.8%)


---
## 3. Load Image Model

Dropout-regularized v1 checkpoint, same architecture as `train_classifier.ipynb`.

In [36]:
def build_model(num_classes, device):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(2048, num_classes)
    )
    m.to(device)
    return m


MODEL_PATH = CKPT_DIR / "v1" / "resnet50_product_category.pt"
assert MODEL_PATH.exists(), f"Checkpoint not found: {MODEL_PATH}"

model = build_model(len(CLASS_NAMES), DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

print(f"Loaded model from {MODEL_PATH}")
print(f"FC layer: {model.fc}")

Loaded model from c:\Projects\cosmetic-expiry-scanner\backend\data\models\v1\resnet50_product_category.pt
FC layer: Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=2048, out_features=3, bias=True)
)


---
## 4. Image-Only Baseline Predictions

Get per-example predictions (predicted class + softmax confidence) for every train-dev image.
This is the reference baseline that fusion will be compared against.

**Determinism note:** `ProductDataset(split="train")` applies *stochastic* augmentations
(RandomAffine/Perspective/Crop/Flip/Rotation/ColorJitter) inside `__getitem__`, so the classic
`train_dev_loader` error rate is random run-to-run (the investigation recorded 28.42% with the
augmented transform). For a reproducible per-example fusion comparison we evaluate with the
same **clean, no-augmentation transform** the investigation itself used for its deterministic
train-dev diagnosis (`CleanEvalDataset` in `train_classifier.ipynb`, ~28.33%). This keeps the
exact same train-dev set, split (seed=42), model, and checkpoint — only the random-augmentation
artifact is removed so the baseline is stable and directly comparable.

In [37]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

# Clean (val-style) transform: plain 224 resize + normalize, no aug.
CLEAN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class CleanEvalDataset(Dataset):
    """Same train-dev images as train_dev_ds, but a deterministic no-aug transform."""
    def __init__(self, base_ds, indices, transform):
        self.samples = [base_ds.samples[i] for i in indices]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_dev_clean_ds = CleanEvalDataset(train_ds, train_dev_idx, CLEAN_TRANSFORM)
train_dev_clean_loader = DataLoader(train_dev_clean_ds, batch_size=32, shuffle=False)
assert len(train_dev_clean_ds) == len(train_dev_ds)
print(f"Clean train-dev loader: {len(train_dev_clean_ds)} images (no augmentation)")

Clean train-dev loader: 1073 images (no augmentation)


In [38]:
baseline_rows = []
batch_idx = 0
model.eval()
with torch.no_grad():
    for imgs, lbls in train_dev_clean_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        confs, preds = probs.max(dim=1)
        for i in range(lbls.size(0)):
            path = train_dev_clean_ds.samples[batch_idx][0]
            fname = Path(path).name
            baseline_rows.append({
                "fname": fname,
                "true_label": CLASS_NAMES[lbls[i].item()],
                "image_pred": CLASS_NAMES[preds[i].item()],
                "image_conf": confs[i].item(),
                "image_correct": (preds[i] == lbls[i]).item(),
            })
            batch_idx += 1

baseline_df = pd.DataFrame(baseline_rows)
print(f"Baseline predictions collected: {len(baseline_df)} examples")

Baseline predictions collected: 1073 examples


In [39]:
baseline_err = 100.0 * (1 - baseline_df["image_correct"].mean())
print(f"Image-only baseline error rate: {baseline_err:.2f}% ({len(baseline_df)} examples)")

# Haircare-specific error rate
hc_mask = baseline_df["true_label"] == "haircare"
if hc_mask.sum() > 0:
    hc_err = 100.0 * (1 - baseline_df.loc[hc_mask, "image_correct"].mean())
    print(f"Haircare error rate (baseline): {hc_err:.2f}% ({hc_mask.sum()} examples)")
else:
    print("No haircare examples in train-dev set.")

# Per-class error breakdown
print("\nPer-class error rates (baseline):")
for cls in CLASS_NAMES:
    cls_mask = baseline_df["true_label"] == cls
    if cls_mask.sum() > 0:
        cls_err = 100.0 * (1 - baseline_df.loc[cls_mask, "image_correct"].mean())
        print(f"  {cls}: {cls_err:.2f}% ({cls_mask.sum()} examples)")

Image-only baseline error rate: 28.33% (1073 examples)
Haircare error rate (baseline): 50.96% (261 examples)

Per-class error rates (baseline):
  haircare: 50.96% (261 examples)
  makeup: 34.12% (85 examples)
  skincare: 19.53% (727 examples)


---
## 5. OCR Extraction + Keyword Matching

> **Kernel note:** this section requires the project venv kernel (`backend/.venv`, Python 3.12),
> which has `google-cloud-vision` installed. The system `C:\Python314` does NOT, so running
> there fails on the `import google.cloud.vision` line.

Runs Google Cloud Vision OCR on each train-dev image, then applies `match_detail()` to
the extracted text. Results are cached incrementally to disk so interrupted runs are
resumable without re-calling the API.

**Failure mode tracking** (three distinct buckets):
- **OCR error**: API call failed (bad credentials, rate limit, malformed image)
- **No text**: OCR ran successfully but found no legible text
- **No strong match**: OCR found text, but no unambiguous product-type keyword hit

The first is a potential systematic issue. The second may correlate with `hard_photo`
bucket images. The third is expected (legible text but no keyword) and falls back to the
image model by design.

**Run strategy:** start with `OCR_N = 15` (sandbox) to confirm OCR is working, then set
`OCR_N = None` for the full 1073-image run. OCR is parallelized (~8 workers) and each
result is written to `ocr_fusion_cache.json` immediately, so an interrupted run resumes
from where it stopped.

In [40]:
from google.cloud import vision as gcvision
from app.core.config import GOOGLE_APPLICATION_CREDENTIALS


def _make_client():
    """Build a Vision client authenticated with the explicit service-account file.

    The bare `ImageAnnotatorClient()` constructor falls back to Application Default
    Credentials (ADC), which are NOT configured locally - so every call would fail with
    'default credentials were not found'. Production vision.py uses the same explicit file.
    """
    return gcvision.ImageAnnotatorClient.from_service_account_json(
        str(GOOGLE_APPLICATION_CREDENTIALS)
    )


def detect_text_local(image_path):
    """Run Google Cloud Vision OCR on a local image file.

    Thin adapter over the Vision API: reads file bytes directly instead of
    downloading from a URL. Returns the same dict shape as vision.py::detect_text().
    Creates a fresh client per call so concurrent threads each get their own.
    """
    content = Path(image_path).read_bytes()
    client = _make_client()
    image = gcvision.Image(content=content)
    result = client.text_detection(image=image)
    if result.error.message:
        raise RuntimeError(f"Vision API error: {result.error.message}")
    raw_text = ""
    if result.text_annotations:
        raw_text = result.text_annotations[0].description
    return {"raw_text": raw_text}


# Quick auth self-check: construct a client to catch bad credentials/config
# BEFORE the expensive loop, so a config problem fails in <1s, not after 1000 img.
try:
    _probe = _make_client()
    print(f"Vision client OK (creds: {GOOGLE_APPLICATION_CREDENTIALS})")
except Exception as e:
    raise RuntimeError(
        f"Failed to init Vision client with credentials at\n"
        f"  {GOOGLE_APPLICATION_CREDENTIALS}\n"
        f"Error: {e}\n\n"
        f"Check the service-account JSON file exists and is valid."
    )

Vision client OK (creds: C:\Projects\cosmetic-expiry-scanner\backend\credentials\shelf-love-353bbeca17ab.json)


In [41]:
OCR_CACHE_PATH = MANIFEST_DIR / "ocr_fusion_cache.json"

# === SANDBOX SLICE TOGGLE ===
# Set OCR_N to a small number (e.g. 15) to validate OCR on a short slice first.
# Set OCR_N = None to process ALL 1073 train-dev images.
OCR_N = None   # <-- change to None for the full run

import threading

# Which images to process this run
all_fnames_total = sorted(train_dev_lookup.keys())
if OCR_N is None:
    worker_fnames = all_fnames_total
    print("Full run: processing ALL", len(worker_fnames), "train-dev images")
else:
    worker_fnames = all_fnames_total[:OCR_N]
    print("SANDBOX slice: processing first", len(worker_fnames), "images")
print("  (set OCR_N = None above to process all 1073)")

# Load existing cache (if any)
if OCR_CACHE_PATH.exists():
    with open(OCR_CACHE_PATH) as f:
        ocr_cache = json.load(f)
    print(f"Loaded existing OCR cache: {len(ocr_cache)} entries")
else:
    ocr_cache = {}
    print("No existing OCR cache found -- starting fresh.")

# Resumability report scoped to the images we intend to process
already_cached = sum(1 for f in worker_fnames if f in ocr_cache)
remaining = len(worker_fnames) - already_cached
print(f"\nResumability: {already_cached}/{len(worker_fnames)} already cached, {remaining} to fetch.")
if remaining == 0:
    print("All target images already OCR'd. Skipping API calls.")

Full run: processing ALL 1073 train-dev images
  (set OCR_N = None above to process all 1073)
Loaded existing OCR cache: 1073 entries

Resumability: 1073/1073 already cached, 0 to fetch.
All target images already OCR'd. Skipping API calls.


In [42]:
import concurrent.futures
import time as _time


def _write_cache_locked(cache_path, cache_dict, lock, retries=8):
    """Atomic write under a lock, tolerant of transient Windows file locks.

    Parallel workers all write to the SAME two paths (tmp + json), and on Windows
    `os.replace` can intermittently fail with `Access is denied` when the destination is
    momentarily locked (AV/Defender scan, another thread's rename). Retry with a short
    backoff so a transient lock aborts the write instead of the whole run.
    """
    with lock:
        tmp = cache_path.with_suffix(".tmp")
        for attempt in range(retries):
            try:
                with open(tmp, "w") as f:
                    json.dump(cache_dict, f, indent=1)
                tmp.replace(cache_path)
                return
            except OSError:
                _time.sleep(0.2 * (attempt + 1))
        # Last resort: write the json directly (skip atomic rename) if all retries fail.
        with open(cache_path, "w") as f:
            json.dump(cache_dict, f, indent=1)


def _classify_entry(raw_text, error):
    """Return (entry_dict, status). status in {'text_found','no_text','error'}."""
    if error is not None:
        return {"raw_text": None, "error": str(error)}, "error"
    if isinstance(raw_text, str) and raw_text.strip():
        return {"raw_text": raw_text, "error": None}, "text_found"
    return {"raw_text": "", "error": None}, "no_text"


def _ocr_one(task):
    """Process one fname -> (fname, entry_dict, status). Runs in a worker thread."""
    fname, img_path = task
    if fname in ocr_cache:
        return fname, ocr_cache[fname], "cached"
    try:
        result = detect_text_local(img_path)
        raw_text = result.get("raw_text", "")
        entry, status = _classify_entry(raw_text, None)
    except Exception as e:
        entry, status = _classify_entry(None, e)
    return fname, entry, status


tasks = [(f, train_dev_lookup[f][0]) for f in worker_fnames \
         if f not in ocr_cache]   # skip already-cached

MAX_WORKERS = 8                       # parallel Google Vision calls
WRITE_EVERY = 25                     # flush cache to disk every N completions
_lock = threading.Lock()
stats = {"done": 0, "text": 0, "blank": 0, "err": 0,
         "hits": sum(1 for f in worker_fnames if f in ocr_cache)}
t0 = _time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_ocr_one, t): t for t in tasks}
    for fut in concurrent.futures.as_completed(futures):
        fname, entry, status = fut.result()
        ocr_cache[fname] = entry
        stats["done"] += 1
        stats[{"text_found": "text", "no_text": "blank", "error": "err"}[status]] += 1
        # Batch the disk write (every N completions + a final flush below). This
        # avoids thousands of full-file renames that trip Windows file locking, while
        # still persisting progress frequently enough that a crash loses little.
        if stats["done"] % WRITE_EVERY == 0:
            _write_cache_locked(OCR_CACHE_PATH, ocr_cache, _lock)
        processed = stats["done"] + stats["hits"]
        if processed % 100 == 0:
            el = _time.time() - t0
            rate = stats["done"] / el if el > 0 else 0
            print(f"  [{processed}/{len(worker_fnames)}] text={stats['text']} "
                  f"no_text={stats['blank']} err={stats['err']} | "
                  f"{rate:.1f} img/s | {el:.0f}s")

# Final flush so the last partial batch is on disk.
if stats["done"] % WRITE_EVERY != 0:
    _write_cache_locked(OCR_CACHE_PATH, ocr_cache, _lock)

print(f"\nOCR loop complete in {_time.time()-t0:.0f}s.")
print(f"  API errors:     {stats['err']}")
print(f"  Text found:     {stats['text']}")
print(f"  No text:        {stats['blank']}")


OCR loop complete in 0s.
  API errors:     0
  Text found:     0
  No text:        0


In [43]:
# Apply match_detail() to every cached OCR result and build per-example OCR table
eval_fnames = list(worker_fnames)   # scope downstream analysis to the processed set
ocr_rows = []
for fname in eval_fnames:
    img_path, true_label = train_dev_lookup[fname]
    entry = ocr_cache.get(fname, {"raw_text": None, "error": None})
    raw_text = entry.get("raw_text")
    error = entry.get("error")

    ocr_category, ocr_keyword, ocr_strong = match_detail(raw_text)

    # Classify the OCR failure mode for downstream analysis
    if error is not None:
        ocr_failure_mode = "error"
    elif raw_text is None or (isinstance(raw_text, str) and not raw_text.strip()):
        ocr_failure_mode = "no_text"
    elif ocr_category is None:
        ocr_failure_mode = "no_match"
    else:
        ocr_failure_mode = "matched"

    ocr_rows.append({
        "fname": fname,
        "true_label": true_label,
        "ocr_text": raw_text,
        "ocr_category": ocr_category,
        "ocr_keyword": ocr_keyword,
        "ocr_strong": bool(ocr_strong) if ocr_strong is not None else False,
        "ocr_failure_mode": ocr_failure_mode,
    })

ocr_df = pd.DataFrame(ocr_rows)
print(f"OCR results table: {len(ocr_df)} examples")
print(f"\nOCR failure mode breakdown:")
print(ocr_df["ocr_failure_mode"].value_counts().to_string())

OCR results table: 1073 examples

OCR failure mode breakdown:
ocr_failure_mode
matched     628
no_match    381
error        55
no_text       9


Merge baseline + OCR results, tag `clear_wrong` from the error-bucketing CSV, and
report overall OCR coverage with all three failure modes kept distinct.

In [44]:
# Merge baseline + OCR results (drop duplicate true_label from ocr_df)
# Scope baseline to the processed set so sandbox runs don't create NaN rows.
ocr_merge = ocr_df.drop(columns=["true_label"])
baseline_scoped = baseline_df[baseline_df["fname"].isin(eval_fnames)].copy()
eval_df = baseline_scoped.merge(ocr_merge, on="fname", how="left")
assert len(eval_df) == len(baseline_scoped), f"Row count mismatch: {len(eval_df)} vs {len(baseline_scoped)}"

# Tag clear_wrong examples from the error bucketing CSV
bucket_csv = MANIFEST_DIR / "traindev_error_bucketing.csv"
bucket_df = None
if bucket_csv.exists():
    bucket_df = pd.read_csv(bucket_csv)
    bucket_df["bucket_fname"] = bucket_df["path"].apply(lambda p: Path(p).name)
    clear_wrong_fnames = set(
        bucket_df.loc[bucket_df["bucket"] == "clear_wrong", "bucket_fname"]
    )
    eval_df["is_clear_wrong"] = eval_df["fname"].isin(clear_wrong_fnames)
    print(f"Tagged {eval_df['is_clear_wrong'].sum()} clear_wrong examples "
          f"(of {len(clear_wrong_fnames)} in bucketing CSV)")
else:
    eval_df["is_clear_wrong"] = False
    print("Warning: traindev_error_bucketing.csv not found. clear_wrong tagging skipped.")

# OCR coverage summary (three failure modes reported separately)
total = len(eval_df)
n_error = (eval_df["ocr_failure_mode"] == "error").sum()
n_no_text = (eval_df["ocr_failure_mode"] == "no_text").sum()
n_no_match = (eval_df["ocr_failure_mode"] == "no_match").sum()
n_matched = (eval_df["ocr_failure_mode"] == "matched").sum()
n_strong = eval_df["ocr_strong"].sum()
print(f"\nOCR coverage on {total} train-dev images:")
print(f"  API errors:              {n_error:4d} ({100*n_error/total:.1f}%)")
print(f"  No text found:           {n_no_text:4d} ({100*n_no_text/total:.1f}%)")
print(f"  Text found, no match:    {n_no_match:4d} ({100*n_no_match/total:.1f}%)")
print(f"  Text found, matched:     {n_matched:4d} ({100*n_matched/total:.1f}%)")
print(f"  Strong keyword match:    {n_strong:4d} ({100*n_strong/total:.1f}%)")

Tagged 17 clear_wrong examples (of 17 in bucketing CSV)

OCR coverage on 1073 train-dev images:
  API errors:                55 (5.1%)
  No text found:              9 (0.8%)
  Text found, no match:     381 (35.5%)
  Text found, matched:      628 (58.5%)
  Strong keyword match:     384 (35.8%)


In [45]:
CLASS_SET = set(CLASS_NAMES)


def apply_fusion(row, conf_threshold=None):
    """Apply fusion rule to a single row.

    Args:
        row: Series with 'image_pred', 'image_conf', 'ocr_category', 'ocr_strong'
        conf_threshold: If set, only override when image_conf < this value.
                       If None, always override on strong match.
    """
    if row["ocr_strong"] and row["ocr_category"] in CLASS_SET:
        if conf_threshold is None or row["image_conf"] < conf_threshold:
            return row["ocr_category"]
    return row["image_pred"]


eval_df["fusion_a"] = eval_df.apply(lambda r: apply_fusion(r), axis=1)
eval_df["fusion_b"] = eval_df.apply(lambda r: apply_fusion(r, conf_threshold=0.6), axis=1)
eval_df["fusion_c"] = eval_df.apply(lambda r: apply_fusion(r, conf_threshold=0.7), axis=1)

# Count overrides per variant
for col, label in [("fusion_a", "Always"), ("fusion_b", "Conf<0.6"), ("fusion_c", "Conf<0.7")]:
    overrides = (eval_df[col] != eval_df["image_pred"]).sum()
    print(f"Variant {label}: {overrides} overrides ({100*overrides/len(eval_df):.1f}% of images)")

Variant Always: 124 overrides (11.6% of images)
Variant Conf<0.6: 51 overrides (4.8% of images)
Variant Conf<0.7: 75 overrides (7.0% of images)


In [46]:
def apply_fusion_loose(row, conf_threshold=None):
    """Override on ANY OCR keyword match (strong or contextual), not just strong."""
    if row["ocr_category"] in CLASS_SET:
        if conf_threshold is None or row["image_conf"] < conf_threshold:
            return row["ocr_category"]
    return row["image_pred"]


eval_df["fusion_d"] = eval_df.apply(lambda r: apply_fusion_loose(r), axis=1)
eval_df["correct_d"] = eval_df["fusion_d"] == eval_df["true_label"]

overrides_d = (eval_df["fusion_d"] != eval_df["image_pred"]).sum()
print(f"Variant Loose-Always: {overrides_d} overrides ({100*overrides_d/len(eval_df):.1f}% of images)")


Variant Loose-Always: 187 overrides (17.4% of images)


In [47]:
eval_df["fusion_e"] = eval_df.apply(lambda r: apply_fusion_loose(r, conf_threshold=0.6), axis=1)
eval_df["correct_e"] = eval_df["fusion_e"] == eval_df["true_label"]

overrides_e = (eval_df["fusion_e"] != eval_df["image_pred"]).sum()
print(f"Variant Loose-Conf<0.6: {overrides_e} overrides ({100*overrides_e/len(eval_df):.1f}% of images)")


Variant Loose-Conf<0.6: 73 overrides (6.8% of images)


In [48]:
additional = eval_df[eval_df["fusion_d"] != eval_df["fusion_a"]]
n_add = len(additional)
n_add_correct = (additional["fusion_d"] == additional["true_label"]).sum()
n_add_incorrect = n_add - n_add_correct

print(f"Additional overrides from loose vs strong-only: {n_add}")
print(f"  Now correct:   {n_add_correct}")
print(f"  Now incorrect: {n_add_incorrect}")
if n_add > 0:
    print(f"  Precision of incremental overrides: {100*n_add_correct/n_add:.1f}%")


Additional overrides from loose vs strong-only: 63
  Now correct:   43
  Now incorrect: 20
  Precision of incremental overrides: 68.3%


In [49]:
eval_df["correct_baseline"] = eval_df["image_pred"] == eval_df["true_label"]
eval_df["correct_a"] = eval_df["fusion_a"] == eval_df["true_label"]
eval_df["correct_b"] = eval_df["fusion_b"] == eval_df["true_label"]
eval_df["correct_c"] = eval_df["fusion_c"] == eval_df["true_label"]

---
## 7. Results

### 7a. Overall Accuracy/Error Comparison

In [50]:
def error_rate(series):
    return 100.0 * (1 - series.mean())


results = []
for col, label in [
    ("correct_baseline", "Image-only (baseline)"),
    ("correct_a",       "+ OCR override (strong match)"),
    ("correct_b",       "+ OCR override (conf < 0.6)"),
    ("correct_c",       "+ OCR override (conf < 0.7)"),
    ("correct_d",       "+ OCR override (loose match, always)"),
    ("correct_e",       "+ OCR override (loose match, conf < 0.6)"),
]:
    err = error_rate(eval_df[col])
    results.append({"Method": label, "Train-dev Error %": round(err, 2)})

results_df = pd.DataFrame(results)
print("=" * 60)
print("Train-Dev Error Rates")
print("=" * 60)
print(results_df.to_string(index=False))
print()

Train-Dev Error Rates
                                  Method  Train-dev Error %
                   Image-only (baseline)              28.33
           + OCR override (strong match)              18.92
             + OCR override (conf < 0.6)              23.77
             + OCR override (conf < 0.7)              21.90
    + OCR override (loose match, always)              16.50
+ OCR override (loose match, conf < 0.6)              22.46



### 7b. Haircare-Only Error Breakdown

In [51]:
hc = eval_df[eval_df["true_label"] == "haircare"]
print(f"Haircare examples in train-dev: {len(hc)}")
if len(hc) > 0:
    hc_results = []
    for col, label in [
        ("correct_baseline", "Image-only (baseline)"),
        ("correct_a",       "+ OCR override (strong match)"),
        ("correct_b",       "+ OCR override (conf < 0.6)"),
        ("correct_c",       "+ OCR override (conf < 0.7)"),
        ("correct_d",       "+ OCR override (loose match, always)"),
        ("correct_e",       "+ OCR override (loose match, conf < 0.6)"),
    ]:
        err = error_rate(hc[col])
        hc_results.append({"Method": label, "Haircare Error %": round(err, 2)})

    hc_results_df = pd.DataFrame(hc_results)
    print(hc_results_df.to_string(index=False))
else:
    print("No haircare examples.")

Haircare examples in train-dev: 261
                                  Method  Haircare Error %
                   Image-only (baseline)             50.96
           + OCR override (strong match)             28.35
             + OCR override (conf < 0.6)             40.23
             + OCR override (conf < 0.7)             36.02
    + OCR override (loose match, always)             26.82
+ OCR override (loose match, conf < 0.6)             40.23


### 7c. `clear_wrong` Bucket Fix Count

In [52]:
cw = eval_df[eval_df["is_clear_wrong"]]
print(f"clear_wrong examples in train-dev: {len(cw)}")
if len(cw) > 0:
    print(f"\nHow many of these does each variant fix?")
    for col, label in [
        ("correct_baseline", "Image-only (baseline)"),
        ("correct_a",       "+ OCR override (strong match)"),
        ("correct_b",       "+ OCR override (conf < 0.6)"),
        ("correct_c",       "+ OCR override (conf < 0.7)"),
        ("correct_d",       "+ OCR override (loose match, always)"),
        ("correct_e",       "+ OCR override (loose match, conf < 0.6)"),
    ]:
        n_fixed = cw[col].sum()
        print(f"  {label:40s}: {n_fixed:2d} / {len(cw)} fixed")

    # Show individual clear_wrong examples that were fixed by any variant
    fixed_by_a = cw[~cw["correct_baseline"] & cw["correct_a"]]
    if len(fixed_by_a) > 0:
        print(f"\nExamples fixed by OCR override (strong match):")
        for _, r in fixed_by_a.iterrows():
            print(f"  {r['fname']}: {r['true_label']} | "
                  f"image={r['image_pred']}({r['image_conf']:.2f}) -> "
                  f"ocr={r['ocr_category']} ({r['ocr_keyword']})")
else:
    print("No clear_wrong examples tagged.")

clear_wrong examples in train-dev: 17

How many of these does each variant fix?
  Image-only (baseline)                   :  8 / 17 fixed
  + OCR override (strong match)           : 12 / 17 fixed
  + OCR override (conf < 0.6)             : 11 / 17 fixed
  + OCR override (conf < 0.7)             : 11 / 17 fixed
  + OCR override (loose match, always)    : 14 / 17 fixed
  + OCR override (loose match, conf < 0.6): 11 / 17 fixed

Examples fixed by OCR override (strong match):
  0810135730082.jpg: haircare | image=skincare(0.45) -> ocr=haircare (conditioner)
  3282779392549.jpg: haircare | image=skincare(0.81) -> ocr=haircare (shampoo)
  0071164343272.jpg: haircare | image=skincare(0.53) -> ocr=haircare (shampoo)
  0870223028675.jpg: skincare | image=haircare(0.85) -> ocr=skincare (body wash)
  3145891812404.jpg: makeup | image=skincare(0.59) -> ocr=makeup (eyeshadow)


### 7d. Summary Table

In [53]:
n_total = len(eval_df)
n_cw = eval_df["is_clear_wrong"].sum()

summary_rows = []
for col, label in [
    ("correct_baseline", "Image-only (baseline)"),
    ("correct_a",       "+ OCR override (strong match)"),
    ("correct_b",       "+ OCR override (conf < 0.6)"),
    ("correct_c",       "+ OCR override (conf < 0.7)"),
    ("correct_d",       "+ OCR override (loose match, always)"),
    ("correct_e",       "+ OCR override (loose match, conf < 0.6)"),
]:
    err = error_rate(eval_df[col])
    hc_err = error_rate(hc[col]) if len(hc) > 0 else float("nan")
    cw_fixed = cw[col].sum() if len(cw) > 0 else 0
    summary_rows.append({
        "Method": label,
        "Train-dev Error %": round(err, 2),
        "Haircare Error %": round(hc_err, 2),
        "clear_wrong fixed": f"{cw_fixed} (of {n_cw})",
    })

summary_df = pd.DataFrame(summary_rows)
print("=" * 90)
print("SUMMARY")
print("=" * 90)
print(summary_df.to_string(index=False))
print()

# OCR coverage context
print(f"OCR coverage: {n_strong}/{n_total} images ({100*n_strong/n_total:.1f}%) "
      f"produced a usable OCR extraction + strong keyword match.")
print(f"  API errors: {n_error} | No text: {n_no_text} | Text, no match: {n_no_match}")

SUMMARY
                                  Method  Train-dev Error %  Haircare Error % clear_wrong fixed
                   Image-only (baseline)              28.33             50.96         8 (of 17)
           + OCR override (strong match)              18.92             28.35        12 (of 17)
             + OCR override (conf < 0.6)              23.77             40.23        11 (of 17)
             + OCR override (conf < 0.7)              21.90             36.02        11 (of 17)
    + OCR override (loose match, always)              16.50             26.82        14 (of 17)
+ OCR override (loose match, conf < 0.6)              22.46             40.23        11 (of 17)

OCR coverage: 384/1073 images (35.8%) produced a usable OCR extraction + strong keyword match.
  API errors: 55 | No text: 9 | Text, no match: 381


In [54]:
best_loose_err = min(error_rate(eval_df["correct_d"]), error_rate(eval_df["correct_e"]))
strong_err = error_rate(eval_df["correct_a"])
if best_loose_err < strong_err:
    print(f"Loose matching improves on strong-only: {strong_err:.2f}% -> {best_loose_err:.2f}%")
else:
    print(f"Strong-only remains best: {strong_err:.2f}% (loose best: {best_loose_err:.2f}%)")


Loose matching improves on strong-only: 18.92% -> 16.50%


### 7e. OCR Failure Mode Breakdown (for interpretation)

In [55]:
print("OCR failure mode distribution on train-dev set:")
print(eval_df["ocr_failure_mode"].value_counts().to_string())
print()

# Correlation with hard_photo bucket from error bucketing CSV
if bucket_csv.exists():
    hard_photo_fnames = set(
        bucket_df.loc[bucket_df["bucket"] == "hard_photo", "bucket_fname"]
    )
    eval_df["is_hard_photo"] = eval_df["fname"].isin(hard_photo_fnames)
    hp = eval_df[eval_df["is_hard_photo"]]
    if len(hp) > 0:
        print(f"\nAmong {len(hp)} hard_photo images:")
        print(hp["ocr_failure_mode"].value_counts().to_string())

OCR failure mode distribution on train-dev set:
ocr_failure_mode
matched     628
no_match    381
error        55
no_text       9


Among 15 hard_photo images:
ocr_failure_mode
matched     13
no_match     2


---
## 8. Save Per-Example Results

In [56]:
OUT_COLS = [
    "fname", "true_label",
    "image_pred", "image_conf", "correct_baseline",
    "ocr_text", "ocr_category", "ocr_keyword", "ocr_strong", "ocr_failure_mode",
    "fusion_a", "correct_a",
    "fusion_b", "correct_b",
    "fusion_c", "correct_c",
    "fusion_d", "correct_d",
    "fusion_e", "correct_e",
    "is_clear_wrong",
]

out_path = MANIFEST_DIR / "ocr_fusion_eval.csv"
eval_df[OUT_COLS].to_csv(out_path, index=False)
print(f"Saved per-example results to {out_path}")
print(f"Columns: {OUT_COLS}")
print(f"Rows: {len(eval_df)}")

Saved per-example results to c:\Projects\cosmetic-expiry-scanner\backend\data\manifests\ocr_fusion_eval.csv
Columns: ['fname', 'true_label', 'image_pred', 'image_conf', 'correct_baseline', 'ocr_text', 'ocr_category', 'ocr_keyword', 'ocr_strong', 'ocr_failure_mode', 'fusion_a', 'correct_a', 'fusion_b', 'correct_b', 'fusion_c', 'correct_c', 'fusion_d', 'correct_d', 'fusion_e', 'correct_e', 'is_clear_wrong']
Rows: 1073
